In [5]:
from dotenv import load_dotenv
load_dotenv()

True

In [6]:
import os
import json
import pandas as pd
from tqdm.auto import tqdm
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import Literal

# --- NEW: Setup paths and import your custom modules ---
sys.path.append(os.path.abspath('..')) 
from modules.ingest import load_faq_data, build_indices
from modules.rag_helper import VectorRAG

# --- NEW: Initialize Client & Data ---
client = OpenAI()

# Load FAQ data and build the winning vector index
documents = load_faq_data()
_, vector_index = build_indices(documents)  # We only need the vector index since it won

# Load the ground truth dataset we generated earlier
gt_path = '../data/ground_truth.json' if os.path.exists('../data/ground_truth.json') else 'data/ground_truth.json'
with open(gt_path, 'r') as f:
    ground_truth = json.load(f)

print(f"Setup complete! Loaded {len(ground_truth)} test questions.")
# 1. Define the exact Zoomcamp Pydantic Model
class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

# 2. Define the exact Zoomcamp Judge Instructions and Prompt
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to the original answer.
Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point
Be fair and focus on correctness, not style.
""".strip()

aqa_judge_prompt = """
Question: {question}
Original Answer (ground truth): {answer_orig}
AI Answer: {answer_llm}
""".strip()

# 3. Two RAG Approaches to Evaluate (Strict vs Friendly Prompts)
PROMPT_STRICT = "QUESTION: {question}\nCONTEXT: {context}\nAnswer ONLY using facts in the context. Be brief."
PROMPT_FRIENDLY = "QUESTION: {question}\nCONTEXT: {context}\nYou are Wali's helpful assistant. Answer accurately using context in a warm tone."

rag_strict = VectorRAG(index=vector_index, llm_client=client, prompt_template=PROMPT_STRICT)
rag_friendly = VectorRAG(index=vector_index, llm_client=client, prompt_template=PROMPT_FRIENDLY)

# 4. Evaluation Function using OpenAI SDK's parse method (matches Zoomcamp's structured output goal)
def evaluate_aqa(question, answer_orig, answer_llm, model="gpt-4o-mini"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )
    
    response = client.beta.chat.completions.parse(
        model=model,
        messages=[
            {"role": "system", "content": aqa_judge_instructions},
            {"role": "user", "content": prompt}
        ],
        response_format=AnswerEvaluation,
        temperature=0.0
    )
    return response.choices[0].message.parsed

# 5. Run the Evaluation on a Sample
import random
random.seed(42)
sample_dataset = random.sample(ground_truth, min(30, len(ground_truth)))

strict_results = []
friendly_results = []

print("Running LLM-as-a-Judge Evaluation (Zoomcamp Method)...")
for item in tqdm(sample_dataset, desc="Evaluating Prompts"):
    q = item['user_query']
    expected_ans = item['answer_orig']
    
    # Generate answers
    ans_strict = rag_strict.rag(query=q)
    ans_friendly = rag_friendly.rag(query=q)
    
    # Judge answers
    judge_strict = evaluate_aqa(q, expected_ans, ans_strict)
    judge_friendly = evaluate_aqa(q, expected_ans, ans_friendly)
    
    strict_results.append(1 if judge_strict.score == "good" else 0)
    friendly_results.append(1 if judge_friendly.score == "good" else 0)

# 6. Summary Comparison Table
strict_score = sum(strict_results) / len(strict_results)
friendly_score = sum(friendly_results) / len(friendly_results)

print("\n=== LLM GENERATION EVALUATION RESULTS ===")
eval_df = pd.DataFrame([
    {"Prompt Template": "Strict (Brief & Factual)", "Good Responses (%)": f"{strict_score * 100:.1f}%"},
    {"Prompt Template": "Friendly (Warm & Polite)", "Good Responses (%)": f"{friendly_score * 100:.1f}%"}
])
print(eval_df.to_string(index=False))

Building Keyword Index...
Building Vector Index...
Both indices built successfully!
Setup complete! Loaded 115 test questions.
Running LLM-as-a-Judge Evaluation (Zoomcamp Method)...


Evaluating Prompts:   0%|          | 0/30 [00:00<?, ?it/s]


=== LLM GENERATION EVALUATION RESULTS ===
         Prompt Template Good Responses (%)
Strict (Brief & Factual)              83.3%
Friendly (Warm & Polite)              76.7%
